In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load CIFAR-10
transform = transforms.Compose([
    transforms.Resize(64),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)f
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Load pretrained ResNet18
model = models.resnet18(weights='IMAGENET1K_V1')

# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Replace final layer for 10 classes
model.fc = nn.Linear(512, 10)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

def train(model, train_loader, criterion, optimizer, epoch):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        if batch_idx % 100 == 0:
            print(f"Epoch {epoch} | Batch {batch_idx}/{len(train_loader)} | "
                  f"Loss: {running_loss/(batch_idx+1):.3f} | "
                  f"Acc: {100.*correct/total:.1f}%")

def evaluate(model, test_loader, criterion):
    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    print(f"\nTest Loss: {test_loss/len(test_loader):.3f} | "
          f"Test Accuracy: {100.*correct/total:.1f}%\n")

# --- PHASE 1: Frozen backbone ---
print("Phase 1: Training with frozen backbone...\n")
for epoch in range(1, 6):
    train(model, train_loader, criterion, optimizer, epoch)
    evaluate(model, test_loader, criterion)

print("Phase 1 complete. Note your test accuracy.\n")

# --- PHASE 2: Unfrozen backbone ---
for param in model.parameters():
    param.requires_grad = True

optimizer = optim.Adam(model.parameters(), lr=0.0005)

print("Phase 2: Training with unfrozen backbone...\n")
for epoch in range(1, 6):
    train(model, train_loader, criterion, optimizer, epoch)
    evaluate(model, test_loader, criterion)

print("Phase 2 complete.")

Using device: cuda
Phase 1: Training with frozen backbone...

Epoch 1 | Batch 0/1563 | Loss: 2.522 | Acc: 18.8%
Epoch 1 | Batch 100/1563 | Loss: 1.781 | Acc: 37.8%
Epoch 1 | Batch 200/1563 | Loss: 1.587 | Acc: 45.4%
Epoch 1 | Batch 300/1563 | Loss: 1.493 | Acc: 49.1%
Epoch 1 | Batch 400/1563 | Loss: 1.429 | Acc: 51.1%
Epoch 1 | Batch 500/1563 | Loss: 1.392 | Acc: 52.4%
Epoch 1 | Batch 600/1563 | Loss: 1.363 | Acc: 53.4%
Epoch 1 | Batch 700/1563 | Loss: 1.333 | Acc: 54.4%
Epoch 1 | Batch 800/1563 | Loss: 1.312 | Acc: 55.1%
Epoch 1 | Batch 900/1563 | Loss: 1.295 | Acc: 55.7%
Epoch 1 | Batch 1000/1563 | Loss: 1.280 | Acc: 56.2%
Epoch 1 | Batch 1100/1563 | Loss: 1.268 | Acc: 56.7%
Epoch 1 | Batch 1200/1563 | Loss: 1.256 | Acc: 57.2%
Epoch 1 | Batch 1300/1563 | Loss: 1.246 | Acc: 57.4%
Epoch 1 | Batch 1400/1563 | Loss: 1.241 | Acc: 57.6%
Epoch 1 | Batch 1500/1563 | Loss: 1.233 | Acc: 57.9%

Test Loss: 1.108 | Test Accuracy: 63.6%

Epoch 2 | Batch 0/1563 | Loss: 1.428 | Acc: 53.1%
Epoch 2 | 